In [ ]:
/*
================================================================================
  MULTI-REGION ARCHITECTURE — EXPLAINED LIKE WE'RE JUST TALKING ABOUT IT
================================================================================

  Real thing that actually happened: in 2017, ONE data center (AWS's
  US-East-1 region) had an outage, and a shockingly large chunk of the
  INTERNET went down with it — because so many companies only ran their
  stuff in that one place. In 2021, a single misconfiguration at
  Facebook took down Facebook, Instagram, AND WhatsApp — all at once,
  worldwide, for six hours.

  ANALOGY: Imagine your entire company operates out of ONE office
  building. It's a great building! But if that ONE building loses
  power, floods, or the internet line to it gets physically cut by a
  backhoe down the street — your ENTIRE company is offline. Doesn't
  matter how good your building is, if it's your ONLY building.

  "Multi-region architecture" just means: don't put all your servers
  in ONE physical location. Spread them across multiple, genuinely
  separate geographic areas, so if a disaster (earthquake, power
  outage, cut cable) takes out ONE, the others can keep things running.

  BUT — and this is important — multi-region isn't JUST about disaster
  recovery. There's a second, equally big reason: SPEED. Physics is
  physics. A signal traveling from Tokyo to Virginia and back takes
  about 150 milliseconds MINIMUM, just from the distance involved —
  before your app even does any actual work. If all your servers are
  in Virginia, EVERY Tokyo user pays that penalty on EVERY request.
  Put a server IN or NEAR Tokyo, and that user gets a much faster
  experience.

  The catch: multi-region is genuinely HARD. Your data now needs to
  exist in multiple places, which opens up a can of worms — replication
  delays, conflicting updates, and failover processes that need to
  actually WORK when you need them (not just look good on a slide).
  Get it wrong, and you can end up WORSE than a single-region setup:
  lost data, "split-brain" (explained later — it's bad), or something
  slower AND less reliable than what you started with.


--------------------------------------------------------------------------------
  1. WHY WOULD YOU EVEN GO MULTI-REGION? (4 different reasons)
--------------------------------------------------------------------------------

  The REASON you need multi-region actually determines WHICH approach
  you should use. So let's go through the 4 main reasons first.

  1.1 DISASTER RECOVERY — "what if the whole building burns down?"

    ANALOGY: Keeping an important document ONLY in your house is risky
    — if there's a house fire, it's gone forever. Keeping a copy in a
    bank safety deposit box (in a totally different building) means a
    fire at your house doesn't destroy everything.

    If your app only runs in ONE region, and that region has an
    outage, EVERYONE is down until it's fixed. With multiple regions,
    a backup region can pick up the slack — but ONLY if it actually
    has enough capacity, fresh enough data, and a TESTED process to
    switch traffic over. (A backup plan you've never actually tested
    is basically a hope, not a plan.)

    Two terms worth knowing here:
      - RTO (Recovery Time Objective): how FAST do you need to recover?
      - RPO (Recovery Point Objective): how much DATA LOSS is
        acceptable if disaster strikes?

        SINGLE REGION (risky):
          Users ──▶ Region A ──[disaster hits]──▶ CRITICAL OUTAGE

        MULTI-REGION (resilient):
          Users ──▶ Region A     Region B
                        │            │
                  [disaster hits]    │
                        │            ▼
                        └────▶ Region B keeps serving traffic!

  1.2 LATENCY REDUCTION — "make it FEEL fast for people far away"

    Speed of light in fiber optic cable isn't infinite — it's a hard
    physical limit. Round trips add up FAST over long distances:

        Route                 | Distance   | Typical round-trip time
        ------------------------|--------------|---------------------------
        Tokyo → Virginia         | 11,000 km    | ~150ms
        Tokyo → Singapore         | 5,300 km     | ~70ms
        Tokyo → Tokyo (local!)     | Local        | ~5ms

    For everyday stuff, 150ms might barely be noticeable. But for
    real-time things — video calls, online gaming, stock trading — that
    difference is the line between "smooth" and "unusable." Putting a
    server geographically CLOSER to your users often matters MORE than
    disaster recovery for a lot of real products.

  1.3 REGULATORY COMPLIANCE — "some data legally has to stay put"

    ANALOGY: Some countries require certain sensitive documents (like
    medical or financial records) to physically stay within that
    country's borders, kind of like how some items can't legally cross
    certain international borders no matter how convenient it'd be to
    ship them.

    Some places have laws about where certain data (health records,
    financial info, personal data) can be stored or processed.
    Multi-region can let you keep EU user data in the EU, keep certain
    financial data in-country, etc. — instead of funneling everyone's
    data through one global location that might violate local rules.

    (To be clear: this is genuinely a legal question, not just an
    engineering one — a real system should involve actual legal/
    compliance input, not just an engineer's best guess.)

  1.4 SCALABILITY — "spread out load that naturally follows the sun"

    ANALOGY: A single, giant restaurant sized to handle its busiest
    possible moment ever sits mostly empty the REST of the time. Way
    more efficient to have several smaller restaurants that each get
    busy at DIFFERENT times of day.

    Some apps have traffic that follows geography — busy in Asia during
    Asian business hours, busy in the US during US hours. A single
    region has to be sized for the COMBINED peak of everyone, which
    wastes a lot of capacity during off-hours. Multiple regions let
    each one scale to ITS OWN local demand instead.


--------------------------------------------------------------------------------
  2. THE 4 MAIN ARCHITECTURAL PATTERNS
--------------------------------------------------------------------------------

  Each pattern trades off complexity, cost, latency, and consistency
  differently. Picking the RIGHT one for your situation matters way
  more than knowing every tiny detail of any single one.

  2.1 ACTIVE-PASSIVE (a.k.a. "Primary-Standby") — the simplest option

    ANALOGY: A backup generator sitting quietly in your basement. It
    does NOTHING most of the time. But the moment the power grid goes
    out, it kicks in and keeps the lights on.

        Primary Region (ACTIVE)              Standby Region (PASSIVE)
              │                                       │
        App Servers (busy) ──async replication──▶ App Servers (idle)
              │                                       │
        Primary DB ─────────────────────────▶  Replica DB
              │
        [Global Load Balancer sends ALL traffic here]

              (if Primary fails → traffic switches to Standby,
               which then becomes the new Primary)

    HOW IT WORKS:
      1. ALL traffic goes to the primary region
      2. Data replicates to the standby region in the BACKGROUND (async)
      3. If the primary fails, DNS/load balancer switches everyone to
         the standby
      4. The standby now becomes the new primary

    Pros                                   | Cons
    -----------------------------------------|--------------------------------------
    Simplest pattern to build                 | Standby resources sit idle most of the time
    No write conflicts — only ONE writer       | Failover takes real time (often minutes,
    Cheaper — standby can be minimal until    |   due to DNS propagation)
    actually needed                           | Possible data loss during failover (whatever
                                              |   hadn't replicated yet is gone)

    BEST FOR: apps that can tolerate a few minutes of downtime and
    SOME data loss during a true regional disaster. A solid STARTING
    point if you don't yet need super-low global latency.

  2.2 ACTIVE-ACTIVE (a.k.a. "Multi-Primary") — everyone's working at once

    ANALOGY: Instead of ONE 24-hour convenience store trying to serve
    the entire country, you open several fully-staffed stores in
    different cities, and each one independently handles its own local
    customers, all at the same time.

        Region US            Region EU            Region APAC
        (App + DB)  ◀──replicate──▶  (App + DB)  ◀──replicate──▶  (App + DB)
            ▲                            ▲                            ▲
        US Users                    EU Users                    APAC Users
       (routed here)              (routed here)               (routed here)

    HOW IT WORKS:
      1. Traffic routes to whichever region is geographically nearest
      2. EACH region can handle BOTH reads AND writes
      3. Data syncs between all regions (usually with "eventual
         consistency" — meaning it'll get consistent EVENTUALLY, not
         instantly)
      4. If one region fails, traffic shifts to the OTHER already-active
         regions (assuming they have spare capacity)

    Pros                                    | Cons
    ------------------------------------------|------------------------------------
    Best latency — users hit a nearby region   | Complex data syncing + conflict handling
    for EVERYTHING, reads AND writes           | Write conflicts ARE possible (same data
    Faster failover — other regions are        |   edited in 2 places at once)
    ALREADY serving live traffic               | Most expensive — every region needs
    Great resource usage — every region is     |   full capacity, not just "backup" capacity
    doing REAL work, not just sitting idle     |

    BEST FOR: truly global apps that genuinely need low-latency reads
    AND writes everywhere, with a data model that can handle (or
    resolve) conflicts. Powerful, but honestly often OVERKILL unless
    your requirements really demand it.

  2.3 READ LOCAL, WRITE GLOBAL — a nice middle ground

    ANALOGY: Think of a big company with branch offices EVERYWHERE that
    can each show you your account info instantly (reading), but any
    actual CHANGE to your account (like closing it) has to go through
    corporate headquarters (writing) — just one authoritative place.

        Region APAC          Region EU            Region US (PRIMARY)
       (read replica)      (read replica)       (Primary DB: reads+writes)
            ▲                    ▲                       ▲    │
        reads served          reads served           reads     │
        locally, fast         locally, fast          served    │
                                                      locally    ▼
                                                            writes forwarded
                                                            here from EVERYWHERE
                                                            (then replicated
                                                             back out to APAC/EU)

    HOW IT WORKS:
      1. Every region has a local READ-ONLY copy of the data
      2. Reads are served LOCALLY — nice and fast
      3. Any WRITE gets forwarded to the ONE primary region
      4. The primary then replicates that change out to all the read
         copies

    Pros                                | Cons
    --------------------------------------|-----------------------------------------
    Simple consistency — only ONE writer   | Writes are SLOW for anyone far from the
    Fast reads, everywhere                | primary region (they have to travel there!)
    No write conflicts to untangle         | The primary is still a single point of
                                          |   failure for writes, unless you have a
                                          |   real failover plan
                                          | "I just wrote something, why don't I see
                                          |   it yet?" — classic read-after-write bug

    BEST FOR: read-heavy apps — social feeds, content sites, e-commerce
    catalogs. If your app is like 90%+ reads, this gets you most of the
    latency win of active-active, for way less complexity.

  2.4 PARTITIONED BY GEOGRAPHY — everyone's data lives where they live

    ANALOGY: Instead of every country's citizens' records being stored
    in ONE giant global filing cabinet, each COUNTRY keeps its own
    citizens' records locally. A German citizen's file lives in
    Germany. A Japanese citizen's file lives in Japan. Simple, and
    naturally satisfies "keep our data in our own country" rules.

        Region US               Region EU               Region APAC
       (US users' DATA)        (EU users' DATA)        (APAC users' DATA)
            ▲                        ▲                         ▲
        US Users                 EU Users                 APAC Users
       (routed here,           (routed here,             (routed here,
        their data lives        their data lives          their data lives
        HERE permanently)       HERE permanently)         HERE permanently)

    HOW IT WORKS:
      1. Each user gets assigned a "home region" (usually based on
         where they signed up / live)
      2. ALL of their data lives in that ONE home region
      3. Their requests almost always route straight to their home region
      4. Most operations never need to cross regions at all

    Pros                                  | Cons
    ----------------------------------------|-------------------------------------
    Little to no replication lag for local   | Cross-region interactions get tricky —
    data — it's just... local                |   what if a EU user messages a US user?
    Naturally satisfies data-residency laws   | If a region fails, EVERYONE whose "home"
    Regions are pretty independent for local  |   is that region is fully down, not just
    operations                               |   partially affected
                                            | Moving countries (user relocates) is messy

    BEST FOR: apps where data is naturally tied to a specific place —
    ride-sharing, local marketplaces, location-based games. If your
    data is ALREADY geography-shaped, this is simpler than replicating
    everything everywhere.


--------------------------------------------------------------------------------
  3. DATA REPLICATION — HOW DO YOU KEEP DATA CORRECT ACROSS REGIONS?
--------------------------------------------------------------------------------

  This is the genuinely hard part. Once your regions can't perfectly,
  instantly talk to each other (which, across the globe, they never
  fully can), you have to decide: how much am I willing to slow down
  writes, in exchange for making sure data is always perfectly
  consistent everywhere?

  3.1 ASYNCHRONOUS REPLICATION — "write locally, sync later"

        Client         Region 1 (Primary)        Region 2
          │                    │                     │
          │──Write Request────▶│                     │
          │                    │──write to local DB   │
          │◀───Success! ✓──────│                     │
          │                    │──replicate (async)──▶│
          │                                       (data arrives HERE, a bit later)

    Pros                                  | Cons
    ----------------------------------------|-------------------------------------
    Low write latency — client doesn't wait  | Possible DATA LOSS if primary fails
    for the OTHER region to confirm anything |   before it finishes replicating
    Primary never blocks on slow/unreachable | "I just wrote this, why don't I see it
    replicas                                 |   yet?" (read-your-writes problem)
    Works fine even across high-latency links| Replicas can serve genuinely STALE data

    Most multi-region setups actually use ASYNC replication, because
    the latency cost of waiting for confirmation across huge distances
    is usually just too painful for users.

  3.2 SYNCHRONOUS REPLICATION — "don't say 'done' until it's SAFE everywhere"

        Client        Region 1              Region 2
          │               │                     │
          │──Write───────▶│                     │
          │               │──write locally       │
          │               │──replicate (sync)───▶│
          │               │                     │──write locally
          │               │◀────ack─────────────│
          │◀──Success! ✓──│

    Pros                                    | Cons
    ------------------------------------------|-------------------------------------
    Much lower risk of data loss (confirmed    | High write latency (every write pays a
    durable in 2+ places before "success")     |   full cross-region round trip, 150ms+
    Strong consistency across regions           |   for distant regions!)
    Can guarantee read-your-writes IF reads     | Reduced availability — writes might need
    are routed carefully                        |   MULTIPLE regions reachable to succeed

    This is rarely used for user-facing writes across genuinely DISTANT
    regions, because the latency hit is just too painful. It's more
    realistic when regions are geographically close, writes are
    low-volume, or you specifically need strong guarantees for a small
    piece of your system.

  3.3 SEMI-SYNCHRONOUS REPLICATION — a practical middle ground

    ANALOGY: You keep an important physical document in a fireproof
    safe AT HOME (nearby, fast to access) AND mail a photocopy to a
    friend across the country (slower, but protects against a total
    house disaster). You don't wait for the mail to arrive before
    considering the document "safe" — the fireproof safe already
    covers you locally.

        Primary (US-East)
              │
        [SYNC]  → Sync Replica (US-West) — nearby, confirmed before "success"
              │
        [ASYNC] → Async Replica (EU)     — catches up eventually
        [ASYNC] → Async Replica (APAC)   — catches up eventually

    Pros                                   | Cons
    -----------------------------------------|-------------------------------------
    Write latency stays tolerable (the sync   | More complex to configure and monitor
    replica is NEARBY, not across the globe)  | Distant regions CAN serve stale reads
    Lower risk of data loss for confirmed       | Still need a plan for "what if even the
    writes (durable in 2 places)              |   NEARBY sync replica goes down?"
    Distant regions still get eventual reads

    This is a genuinely COMMON real-world pattern: protect writes
    synchronously within a NEARBY pair of regions, and let far-away
    regions catch up asynchronously for fast local reads.

  3.4 HANDLING CONFLICTS IN ACTIVE-ACTIVE (when 2 regions BOTH accept writes)

    If multiple regions can ALL accept writes, sometimes two different
    users in two different regions edit the SAME piece of data at
    roughly the SAME time, without either region knowing about the
    other's edit yet. Now what?

    OPTION 1 — LAST-WRITER-WINS (LWW): whichever write has the LATEST
    timestamp just... wins. Simple, but can SILENTLY throw away someone's
    update. Fine for stuff like "last known location" or caching,
    where losing an update isn't a big deal.

    OPTION 2 — CRDTs (Conflict-free Replicated Data Types): specially
    designed data structures that can be mathematically MERGED without
    any conflict at all — no arbitrary "winner" needed.

        Data type   | CRDT type      | How it resolves
        --------------|-------------------|-----------------------------------
        Counter       | G-Counter          | Just add up all the increments
        Set           | OR-Set             | Union of everything added
        Register      | LWW-Register       | Latest timestamp wins (same idea as above)
        Map           | OR-Map             | Merge entry-by-entry, by key

    OPTION 3 — APPLICATION-LEVEL RESOLUTION: let your actual BUSINESS
    LOGIC decide how to merge things, based on what makes sense for
    YOUR specific situation (not a generic algorithm).

    THE HONEST LIMIT: CRDTs are great for simple things — counters,
    sets, basic key-value data. For genuinely COMPLEX business logic
    (like inventory management or financial transactions), you usually
    need custom application-level resolution — or you just avoid the
    problem entirely by routing all writes for that data to ONE region.


--------------------------------------------------------------------------------
  4. TRAFFIC ROUTING — HOW DO USERS FIND "THEIR" REGION?
--------------------------------------------------------------------------------

  4.1 GEODNS — "DNS gives different answers based on WHERE you're asking from"

    ANALOGY: Calling a company's "local" phone number, and depending
    on your area code, the call automatically gets routed to the
    nearest branch office, without you doing anything special.

        User in US ──DNS query──▶ GeoDNS ──▶ returns US region's IP
        User in EU ──DNS query──▶ GeoDNS ──▶ returns EU region's IP
        User in Asia──DNS query──▶ GeoDNS ──▶ returns APAC region's IP

    Pros                                  | Cons
    ----------------------------------------|-------------------------------------
    Simple to set up (Route 53, Cloudflare,  | DNS gets CACHED — if a region fails,
    Google Cloud DNS, etc.)                  |   cached answers can still point there
    No extra network hop — users connect      | Failover is SLOW (limited by DNS
    directly to their assigned region        |   "time to live," often 60-300 seconds)
    Works with any backend                    | Based on the DNS RESOLVER's location,
                                             |   which isn't always the USER's actual
                                             |   location (public DNS like 8.8.8.8 can
                                             |   route people to the "wrong" region)

  4.2 GLOBAL LOAD BALANCER (ANYCAST) — "one IP address, announced everywhere"

    ANALOGY: Imagine ONE emergency phone number that, depending on
    which CITY you're calling from, automatically connects you to the
    closest dispatch center — even though everyone dialed the EXACT
    SAME number.

        Users Worldwide ──▶ SAME IP: 1.2.3.4 ──▶ routed by internet
                                                   topology to nearest
                                                   PoP (US, EU, or APAC)

    The internet's own routing infrastructure (BGP) sends each user's
    traffic to whichever announced location is "closest" on the actual
    NETWORK — which isn't always the same as geographically closest.

    Pros                                | Cons
    --------------------------------------|-----------------------------------
    Often FASTER failover than DNS changes  | Requires BGP control (not every
    Single IP simplifies client setup       |   provider offers this)
    Follows real network topology, not       | Best for STATELESS services —
    just geography                          |   handing off an active connection
                                           |   mid-failover is tricky
                                           | Harder to debug (can't always predict
                                           |   which location a user will hit)

  4.3 APPLICATION-LEVEL ROUTING — "a smart gateway decides, based on business logic"

    ANALOGY: A receptionist who actually LOOKS at your appointment
    details before directing you — "oh, you're here for billing, that's
    the 3rd floor" — instead of just sending everyone to the same place.

        User Request ──▶ Global Gateway ──▶ decides based on:
                                              - user's HOME region (from profile)
                                              - request TYPE (reads → nearest replica,
                                                writes → primary)
                                              - CURRENT LOAD across regions
                                              - feature flags / A/B tests

    Pros                                | Cons
    --------------------------------------|-----------------------------------
    Maximum flexibility for complex logic   | Adds latency (every request passes
    Business-logic-aware routing (fits      |   through the gateway first)
    naturally with geo-partitioning)        | More complex to build and run
    Can treat different request TYPES        | The gateway itself needs to be
    differently                             |   globally distributed, or it becomes
                                           |   its own bottleneck

  4.4 LATENCY-BASED ROUTING — "send them wherever is ACTUALLY fastest right now"

    ANALOGY: Instead of assuming "closest on a map = fastest," you
    actually TIME how long it takes to get a response from each
    option, and pick whichever one is genuinely quickest RIGHT NOW —
    since real road/network conditions don't always match the map.

        User ──▶ measured: Region A = 50ms, Region B = 120ms, Region C = 80ms
                                    │
                                    ▼
                          Route to Region A (lowest measured latency)

    Pros                               | Cons
    -------------------------------------|-----------------------------------
    Optimal routing based on REAL         | Needs ongoing latency-measurement
    measurements, not assumptions          |   infrastructure
    Accounts for actual network conditions | Results can shift over time
    Better than geography alone for        | Adds real operational complexity
    "weird" network topology cases          |

    A user in Australia might actually get lower latency to US-West
    than to nearby Singapore, purely because of how undersea cables are
    routed — geography can genuinely lie to you here.


--------------------------------------------------------------------------------
  5. HANDLING FAILURES AND FAILOVERS
--------------------------------------------------------------------------------

  Having multiple regions only actually HELPS if your system can
  DETECT a failure and ROUTE AROUND it, automatically or otherwise.

  5.1 HEALTH CHECKS — "how do you even know a region is unhealthy?"

        Health Checker ──checks /health──▶ Region US    ✓ Healthy
                       ──checks /health──▶ Region EU     ✓ Healthy
                       ──checks /health──▶ Region APAC   ✗ Unhealthy
                                                │
                                          (excluded from Load Balancer)

    THINGS TO GET RIGHT:
      - Check from MULTIPLE locations: a single failed check might just
        mean a network hiccup between the checker and that region —
        not that the region itself is actually down. Require agreement
        from at least a MAJORITY of checkers.
      - Check DEEP health, not just "did the server say HTTP 200?" —
        actually verify it can reach its database and truly serve requests.
      - Require SEVERAL consecutive failures before acting, to avoid
        "flapping" (frantically switching back and forth) due to brief
        blips.
      - Balance sensitivity vs. stability: too twitchy, and you get
        FALSE failovers. Too relaxed, and REAL failures take too long
        to catch.

  5.2 FAILOVER STRATEGIES

    AUTOMATIC FAILOVER: the system detects a failure and switches
    traffic over BY ITSELF, no human needed.

        Traffic ──▶ Region A (Primary) ──[failure detected]──▶
                                              Traffic ──▶ Region B (takes over)

      Pros: fast recovery, no waiting on a human to wake up
      Cons: can trigger FALSELY (which can actually cause MORE damage
      than the original problem!), and risks "split-brain" (next
      section) if not done VERY carefully.

    MANUAL FAILOVER: a human decides, after actually investigating.

      Pros: avoids false alarms, human judgment can catch weird edge cases
      Cons: SLOW, needs someone actually on-call and awake, and
      decisions made under 3am panic-pressure can be error-prone.

    HYBRID (most common in practice): automation handles the
    WELL-UNDERSTOOD failure cases, but a human can OVERRIDE or pause it
    if something looks off.

  5.3 FAILBACK — bringing a recovered region back online (harder than it sounds!)

    CHALLENGES:
      - DATA DIVERGENCE: while that region was down, writes kept
        happening ELSEWHERE — so the recovered region's data is now
        genuinely OUT OF DATE
      - Don't just slam 100% of traffic back onto it immediately —
        what if it's not actually fully healthy yet?
      - VERIFY it's truly ready, not just "responding to health checks"

    BEST PRACTICE — GRADUAL FAILBACK (just like a canary rollout):

        Step 1: send it 10% of traffic → watch closely
        Step 2: increase to 50%          → watch closely
        Step 3: finally go to 100%       → done

    This catches problems while they're only affecting a SMALL slice
    of users, instead of everyone at once.

  5.4 SPLIT-BRAIN — the nightmare scenario

    ANALOGY: Imagine two branch managers of the same bank, cut off from
    communicating with each other (phone lines down), and BOTH decide
    "well, I guess I'm in charge now" — and both start independently
    approving loans and moving money, unaware the other is doing the
    exact same thing. When communication is restored, you have TWO
    conflicting sets of "the truth" that somehow need to be reconciled.

        Region A: "I am Primary!"  ✗cannot talk to✗  Region B: "I am Primary!"
              │                                              │
        Writes go to A                              Writes go to B
              │                                              │
              └──────────── CONFLICTING DATA! ───────────────┘

    When the network heals, you're left with two datasets that
    disagree, and SOME writes will inevitably get lost trying to
    reconcile them. Not fun for anyone.

    PREVENTION STRATEGIES:
      - QUORUM-BASED DECISIONS: require a MAJORITY of regions to agree
        before anyone can call itself "primary." With 3 regions, you
        need at least 2 to agree. (This is literally how algorithms
        like Raft and Paxos work under the hood.)
      - EXTERNAL ARBITER: use a separate coordination service to decide
        who's primary — but that arbiter ITSELF needs to be highly
        available and OUTSIDE the failure it's arbitrating (otherwise
        you've just moved the problem).
      - FENCING TOKENS: give each "primary" a constantly-increasing
        token number. If an OLD primary tries to write with an outdated
        token, it gets REJECTED — even if that old primary still
        thinks it's in charge.
      - STONITH ("Shoot The Other Node In The Head"): forcefully shut
        down or isolate the OLD primary before letting a new one take
        over. Effective, but needs to be automated very carefully.


--------------------------------------------------------------------------------
  6. COST — MULTI-REGION IS NOT FREE (and can surprise you)
--------------------------------------------------------------------------------

    Cost type      | What it covers
    -----------------|---------------------------------------------------
    Compute           | Running servers in MULTIPLE places, often less
                      |   efficiently utilized than a single big region
    Storage           | Replicated data = 2-3x the storage cost
    Data Transfer     | Cross-region bandwidth — often the SURPRISE bill,
                      |   especially replicating large amounts of data
    Licensing         | Some software/databases charge PER region
    Operations        | More complex monitoring, debugging, and on-call

  WAYS TO KEEP COSTS DOWN:
    - Tiered replication: only sync-replicate the truly CRITICAL data;
      let less-critical stuff replicate async, or not at all
    - Compress data before shipping it across regions
    - Cache heavily at the edge, so you're not constantly querying a
      distant database
    - Reserved/committed capacity for predictable workloads
    - Use active-PASSIVE for non-critical systems — not everything
      needs the expense of full active-active


--------------------------------------------------------------------------------
  7. INTERVIEW / DESIGN GUIDANCE — START SIMPLE, DON'T OVER-ENGINEER
--------------------------------------------------------------------------------

  7.1 START SIMPLE, THEN EVOLVE

    Don't jump straight to full active-active on day one — it's
    expensive and complicated. A sensible progression:

      1. Single region, with GOOD backups — genuinely enough for most
         apps just starting out
      2. Add read replicas in other regions once latency actually
         becomes a real, measured problem
      3. Add active-passive for disaster recovery once your RTO/RPO
         requirements demand it
      4. ONLY graduate to active-active when you have a GENUINE need
         for global write availability, plus an actual conflict
         strategy in place

    Plenty of hugely successful products ran single-region for YEARS
    before ever going multi-region.

  7.2 DESIGN ASSUMING THINGS WILL FAIL

    - Avoid region-SPECIFIC dependencies (if your login system ONLY
      lives in US-East, the WHOLE WORLD goes down when US-East does)
    - Actually TEST your failovers — an untested failover plan is
      basically a guess
    - Write it DOWN (runbooks) — 3am during an actual outage is not
      the time to be figuring things out from scratch
    - Automate recovery where you reasonably can — manual recovery is
      slow and error-prone

  7.3 KNOW YOUR CONSISTENCY GUARANTEES, CLEARLY

    Ask yourself:
      - WHAT data is allowed to be stale, and for HOW long?
      - WHICH operations genuinely need strong consistency?
      - What actually HAPPENS when a user reads stale data?

    If you can't answer these clearly, you don't yet fully understand
    how your own system behaves under failure.

  7.4 METRICS WORTH WATCHING

    - Replication lag (how far behind are the replicas?)
    - Cross-region latency (is traffic actually going to the nearest region?)
    - Regional error rates (is any one region quietly struggling?)
    - Failover frequency AND duration (are you failing over too often?
      Are failovers taking too long when they DO happen?)

  7.5 ACTUALLY PRACTICE YOUR FAILOVERS

    Run drills regularly:
      - Region evacuation drills: can you move ALL traffic out of a
        region quickly, on purpose?
      - Failback procedures: does bringing a region BACK actually work,
        or have you only ever tested taking it DOWN?
      - Split-brain scenarios: what genuinely happens if regions can't
        talk to each other?
      - Recovering from data divergence: can you actually reconcile two
        disagreeing datasets after the fact?

    The absolute worst time to discover your failover doesn't work is
    during a REAL outage.


--------------------------------------------------------------------------------
  8. DECISION GUIDE — WHICH PATTERN SHOULD I ACTUALLY USE?
--------------------------------------------------------------------------------

  A short chain of questions narrows it down fast:

        Do you need genuinely LOW LATENCY, globally?
                    │
          ┌─────────┴─────────┐
          NO                  YES
          │                    │
          ▼                    ▼
    Does data need to     Is your workload
    stay in a specific    WRITE-HEAVY?
    region (compliance)?       │
          │              ┌─────┴─────┐
      ┌───┴───┐           NO          YES
      YES      NO         │            │
      │         │         ▼            ▼
      ▼         ▼    Read Local,   Do writes need
    Geo-      Active-  Write         STRONG
   Partitioned Passive  Global      consistency?
                                        │
                                  ┌─────┴─────┐
                                  YES          NO
                                  │             │
                                  ▼             ▼
                          Single Writer    Active-Active
                          + Sync Standby   (eventual consistency)


--------------------------------------------------------------------------------
  9. QUICK-REFERENCE DECISION CHECKLIST
--------------------------------------------------------------------------------

  Before finalizing a multi-region design, check:

    [ ] Do I actually know WHY I need multi-region — disaster
        recovery, latency, compliance, or scalability? (The reason
        changes which pattern fits!)
    [ ] Have I picked concrete RTO (how fast to recover) and RPO (how
        much data loss is OK) numbers, instead of vague "we'll be fine"?
    [ ] Is my workload mostly READS, mostly WRITES, or a genuine mix?
        (This alone often decides between "read local write global"
        and full active-active.)
    [ ] If multiple regions can write, do I have an ACTUAL conflict
        resolution strategy (LWW, CRDTs, app-level), or am I just
        hoping conflicts won't happen?
    [ ] Have I thought through split-brain prevention (quorum, fencing
        tokens, an external arbiter) — not just "it probably won't happen"?
    [ ] Do I have real, TESTED health checks and a failover process —
        or just an assumption that failover will "just work"?
    [ ] Have I actually planned FAILBACK (bringing a recovered region
        back gradually), not just failover?
    [ ] Have I accounted for the REAL costs — cross-region data
        transfer especially, which is a common surprise line item?
    [ ] Am I starting SIMPLE and evolving as real requirements demand
        it, rather than jumping straight to the most complex pattern
        "just in case"?


--------------------------------------------------------------------------------
  TL;DR — THE ONE THING TO REMEMBER
--------------------------------------------------------------------------------

  Multi-region architecture is fundamentally a set of TRADE-OFFS, not
  a single "correct" answer:

    - Latency vs. Consistency: faster LOCAL reads often mean slightly
      STALE data, unless you add real coordination
    - Availability vs. Complexity: more regions means more moving
      parts — more things that CAN break, and more to operate
    - Cost vs. Resilience: redundancy costs real money — roughly double
      the regions, roughly double the cost

  The best multi-region designs aren't the FANCIEST ones — they're the
  ones where the designer actually understood their specific workload,
  and picked the SIMPLEST pattern that genuinely met their real RTO,
  RPO, latency, and compliance requirements. Don't reach for
  active-active just because it sounds impressive — reach for it
  because your actual requirements demand it.

================================================================================
*/